# Can a longer context help Humanoid stay upright?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccnets-team/causal-gpt-rl/blob/main/examples/can_longer_context_help_humanoid.ipynb)

One MuJoCo bundle drives a Humanoid for a thousand steps. This notebook runs it
three times with the same weights and changes one thing between runs: how many
steps of the past the policy keeps — 8, 32, then 128.

Nothing is retrained in between. Retention is a load-time argument, so the three
runs load the same file and differ by one number.

## Install

`mujoco` is pinned to `3.2.3`. A different simulator release is a different
measurement even with identical weights and seeds, so the numbers below are
defined on this one.

In [ ]:
%pip install -q "causal-gpt-rl[hub,mujoco]" "mujoco==3.2.3"

## Get the repository

The sweep reuses the rollout loop from
[`examples/deploy/reproduce.py`](https://github.com/ccnets-team/causal-gpt-rl/blob/main/examples/deploy/reproduce.py),
which is a file of this repository rather than part of the installed package. On
Colab, clone it; in a checkout, this cell does nothing.

In [ ]:
from pathlib import Path

if not Path("examples/deploy/reproduce.py").is_file():
    !git clone -q https://github.com/ccnets-team/causal-gpt-rl.git
    %cd causal-gpt-rl

## The knob

`kv_cache_max_len` is how many steps of the past a rollout keeps. It defaults to
the bundle's `context_length` — the window the policy was trained on, **32**
here — and that window is not a cap: 128 runs the same policy with four times
the history it ever saw in training.

That is the entire difference between the three runs:

```python
load_runner_from_hub(..., kv_cache_max_len=8)     # a quarter of the window
load_runner_from_hub(..., kv_cache_max_len=32)    # the window it was trained on
load_runner_from_hub(..., kv_cache_max_len=128)   # four times past it
```

In [ ]:
import torch

from examples.deploy.reproduce import installed_versions, print_stack_report

repo_id = "ccnets/causal-gpt-rl"
subfolder = "humanoid-v5"
env_id = "Humanoid-v5"

KV_VALUES = [8, 32, 128]        # the bundle's own context_length is 32

# One seed per episode. The episode count is also the width of the batch the
# policy runs as, and that width is part of the measurement — lowering it for a
# quicker look changes the numbers, it does not just shorten the run.
episodes = 50
max_steps = 1000
seeds = list(range(episodes))
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"{repo_id}/{subfolder} on {env_id}  ({device})\n")
print_stack_report(installed_versions())

## Run the sweep

Each setting gets a fresh policy and a fresh set of environments, and every one
of them runs the same seeds. The three run one after another rather than at
once, so the timing is what a single user would see.

Expect roughly two minutes on a recent GPU. On CPU it is much longer — on Colab,
check that the runtime has a GPU attached before starting.

In [ ]:
import gymnasium as gym
import numpy as np

from causal_gpt_rl.inference import load_runner_from_hub
from examples.deploy.reproduce import run_seed_batch

results = []

for kv in KV_VALUES:
    policy = load_runner_from_hub(
        repo_id=repo_id,
        subfolder=subfolder,
        device=device,
        num_envs=episodes,
        kv_cache_max_len=kv,
    )
    envs = gym.vector.SyncVectorEnv(
        [lambda eid=env_id: gym.make(eid) for _ in seeds],
        autoreset_mode=gym.vector.AutoresetMode.SAME_STEP,
    )
    try:
        returns, lengths, ends = run_seed_batch(
            envs, policy, seeds, max_steps, per_seed=False
        )
    finally:
        envs.close()

    # `terminated` is the Humanoid falling over; `truncated` is the thousand-step
    # limit arriving with it still upright. Only the first one is a failure.
    fell = [int(seeds[row]) for row in np.flatnonzero(ends["terminated"])]
    results.append({
        "kv": kv,
        "returns": returns,
        "full": episodes - len(fell),
        "fell": fell,
    })
    print(f"kv={kv:<4} mean={returns.mean():8.2f}  worst={returns.min():8.2f}  "
          f"{results[-1]['full']}/{episodes} full episodes", flush=True)

## The table

The mean is the least interesting column. What moves is the bottom of the range:
the **worst episode**, and how many rollouts stayed upright for the full thousand
steps instead of falling before the limit arrived.

In [ ]:
print(f"{'kv':>4}  {'return (mean +/- std)':>24}  {'worst episode':>14}  "
      f"{'full episodes':>14}")
for r in results:
    returns = r["returns"]
    full = f"{r['full']}/{episodes}"
    print(f"{r['kv']:>4}  {returns.mean():11.2f} +/- {returns.std():<9.2f}  "
          f"{returns.min():14.2f}  {full:>14}")

print("\nseeds where the Humanoid fell:")
for r in results:
    print(f"  kv={r['kv']:<4} {r['fell'] or 'none'}")

## What we measured

Notebooks here are committed without their outputs, so this is what the cells
above printed on our machine — an RTX 4080 Laptop GPU, MuJoCo 3.2.3,
Gymnasium 1.2.3, torch 2.8.0, against revision `111b955` of the bundle.

| kv | return (mean +/- std) | worst episode | full episodes |
|---:|---:|---:|---:|
| 8 | 7617.62 +/- 1570.12 | 505.03 | 45/50 |
| 32 | 7809.04 +/- 1128.29 | 793.72 | 47/50 |
| 128 | **8040.63 +/- 121.10** | **7334.43** | **48/50** |

The mean moves by 5%. The worst episode moves by a factor of fourteen, and the
spread collapses from +/- 1570 to +/- 121.

**A short memory does not make the policy walk worse. It makes it fall over
sometimes.** At `kv=8` five of the fifty seeds ended early; at `kv=128`, two did.

Longer retention was close to free here: the three runs took 41, 47, and 51
seconds, and the difference in memory use was too small to measure against what
the machine was already using.

## What this shows, and what it does not

**One bundle, one environment.** Humanoid gains from more history. Across the
published bundles retention helps some and hurts others, so this is a
measurement to repeat in your own environment, not a setting to copy.

**Longer is not automatically better.** What bounds retention is the episode: a
cache only holds the steps that have actually run, so with `max_steps=1000` any
value above 1000 behaves exactly like 1000.

**The batch width is part of the measurement.** Fifty episodes run as one
fifty-row batch, and a batch of one reduces floating point in a different order
— in a closed loop that difference compounds. Changing `episodes` above changes
both the sample and the batch, so a shorter run is a different measurement
rather than a rougher one.

To measure a bundle under the published protocol, or to try another environment:

```
python -m examples.deploy.reproduce --env-id Humanoid-v5 --kv-cache-max-len 128
```

More on what retention is, and why the trained window is not a ceiling:
[Rollout History](https://github.com/ccnets-team/causal-gpt-rl#rollout-history).